In [1]:
import os
from scipy.signal import spectrogram 
from scipy.io import wavfile
import numpy as np
from PIL import Image
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt

In [2]:
#Does not need to run each time 
#parameters for the split audio function 
input_file =r"C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\noise_target数据集\noise_target数据集\noise_未切分\DATA0001\DATA0033.wav"
output_dir='NoiseFragments2'
os.makedirs(output_dir, exist_ok=True)
chunk_duration=3
startingfilenumber=100


In [ ]:
#Does not need to run eah time 
#Function Def for turning larger audio files into smaller 3 second fragments 
import os
import numpy as np
import librosa
import soundfile as sf

def split_audio_into_chunks(input_file, output_dir, chunk_duration=3,startingfilenumber=0):
    os.makedirs(output_dir, exist_ok=True)
    
    # Load audio file
    y, sr = librosa.load(input_file, sr=None)  # keep original sample rate
    
    chunk_samples = chunk_duration * sr
    total_samples = len(y)
    
    num_chunks = total_samples // chunk_samples
    
    for i in range(num_chunks):
        start = i * chunk_samples
        end = start + chunk_samples
        chunk = y[start:end]
        
        output_path = os.path.join(output_dir, f"chunk_{startingfilenumber:04d}.wav")
        sf.write(output_path, chunk, sr)
        startingfilenumber+=1
       
    
    print(f"Saved {num_chunks} chunks to {output_dir}")



In [ ]:
#Does not need to run each time 
#Usage for the split_audio function 
split_audio_into_chunks(input_file, output_dir, chunk_duration,startingfilenumber)

Saved 100 chunks to NoiseFragments2


In [ ]:
#Does not need to run each time
#function for combining folder files into one
import os
import shutil

def combine_folders(source_folders, destination_folder, move_files=False):
    """
    Combines contents of multiple folders into a single folder.
    
    Parameters:
        source_folders (list): List of folder paths to combine
        destination_folder (str): Target folder path
        move_files (bool): If True, moves files. If False, copies files.
    """
    
    os.makedirs(destination_folder, exist_ok=True)
    
    for folder in source_folders:
        for root, _, files in os.walk(folder):
            for file in files:
                source_path = os.path.join(root, file)
                
                # Handle duplicate filenames
                base_name = file
                name, ext = os.path.splitext(base_name)
                counter = 1
                dest_path = os.path.join(destination_folder, base_name)
                
                while os.path.exists(dest_path):
                    new_name = f"{name}_{counter}{ext}"
                    dest_path = os.path.join(destination_folder, new_name)
                    counter += 1
                
                if move_files:
                    shutil.move(source_path, dest_path)
                else:
                    shutil.copy2(source_path, dest_path)

    print("Done combining folders.")


In [ ]:
#Does not need to run each time 
#using combin_folders function
source_dirs = [
    r"C:\Users\yusle\OneDrive\Desktop\boat_audio\NoiseFragments1",
    r"C:\Users\yusle\OneDrive\Desktop\boat_audio\NoiseFragments2"
    
]

combine_folders(source_dirs, "combined_noise", move_files=False)


Done combining folders.


In [11]:
audio_folder=r'C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss'
output_folder='W4_spec'
os.makedirs(output_folder, exist_ok=True)

In [2]:
# We need to create parmeters for the spectrogram
freq_min=10
freq_max=1000
window_size=1024
overlapp=800
n_fft=1024*4  #2^10

In [13]:
#loop for turning audio files in the directed folder into spectrogram with a mask and then normilzing it and saving the image to the output folder
for file in os.listdir(audio_folder):
    file_path= os.path.join(audio_folder,file)
    print(file_path)
    fs,x=wavfile.read(file_path)
    f,t,S=spectrogram(x,fs,nperseg=window_size,noverlap=overlapp,nfft=n_fft)
    f_mask=(f>=freq_min)&(f<=freq_max)
    sxx=S[f_mask,:]
    G=10*np.log10(sxx+1e-8)
    G=np.flipud(G)
    # normalize the spectrogram
    norm=Normalize(vmin=np.min(G),vmax=np.max(G))
    G_normalized=norm(G)
    # convert spectrogram to image
    G_colormap=plt.cm.jet( G_normalized)
    G_image=( G_colormap[:,:,:3]*255).astype(np.uint8)
    G_resized=Image.fromarray(G_image).resize((224,224))
    output_file=os.path.join(output_folder,os.path.basename(file_path).replace('.wav','.png'))
    G_resized.save(  output_file)
    
        
    

C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_1.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_10.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_100.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_101.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_102.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_103.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_104.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_105.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_106.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_107.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_108.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_109.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_11.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_110.wav
C:\Users\yusle\OneDrive\Desktop\Whale_audio\Riss\Riss\10_111.wav
C:\Users\yusle\OneDrive\Deskt

In [3]:
# create model for training
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

In [14]:
#for splitting folder with subfolder classes into folder with train and validation 70/30 split folder and each having all the subfolder classes  
import os
import shutil
import random

# ===== CONFIG =====
source_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\whale_spec"   # folder with class subfolders
train_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\whale_spec_split\Train"
test_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\whale_spec_split\Val"
split_ratio = 0.7
random_seed = 42
# ==================

random.seed(random_seed)

# Create Train and Test directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Loop over each class folder
for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    # Create class folders in Train and Test
    train_class_dir = os.path.join(train_dir, class_name)
    test_class_dir = os.path.join(test_dir, class_name)

    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(test_class_dir, exist_ok=True)

    # Get all image files
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    random.shuffle(images)

    split_index = int(len(images) * split_ratio)
    train_images = images[:split_index]
    test_images = images[split_index:]

    # Copy files
    for img in train_images:
        shutil.copy2(
            os.path.join(class_path, img),
            os.path.join(train_class_dir, img)
        )

    for img in test_images:
        shutil.copy2(
            os.path.join(class_path, img),
            os.path.join(test_class_dir, img)
        )

    print(f"{class_name}: {len(train_images)} train / {len(test_images)} test")

print("✅ Dataset split complete.")


W1_spec: 2506 train / 1074 test
W2_spec: 3304 train / 1417 test
W3_spec: 1502 train / 645 test
W4_spec: 2293 train / 984 test
✅ Dataset split complete.


In [4]:
# prepare data
data_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\whale_spec_split"  # Replace with your folder path
# Hyperparameters
batch_size = 16
num_epochs = 10
learning_rate = 0.001
num_classes =4  # Number of subfolders-(each folder is a class)
print(num_classes)

# Image transformations
transform = transforms.Compose([transforms.ToTensor()])

4


In [5]:
# Load dataset
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "Train"), transform=transform)
val_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "Val"), transform=transform)

In [6]:

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [7]:
class ViTModel(nn.Module):
    def __init__(self, num_classes):
        super(ViTModel, self).__init__()
        # Load pretrained Vision Transformer model
        self.model = models.vit_b_16(pretrained=True)
        
        # Replace the head (classification layer)
        in_features = self.model.heads.head.in_features  # Get input features of the head
        self.model.heads.head = nn.Linear(in_features, num_classes)  # Replace with new classification layer

    def forward(self, x):
        return self.model(x)

In [8]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
1
NVIDIA GeForce RTX 3070


In [9]:
class ResNet18Model(nn.Module):
    def __init__(self, num_classes):
        super(ResNet18Model, self).__init__()
        
        self.model = models.resnet18(pretrained=True)
        
        # Replace final fully connected layer
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [10]:
class ResNet50Model(nn.Module):
    def __init__(self, num_classes):
        super(ResNet50Model, self).__init__()
        
        self.model = models.resnet50(pretrained=True)
        
        # Replace final fully connected layer
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [11]:
#declaring the efficentNetModel structure 
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetModel, self).__init__()
        self.model = models.efficientnet_b0(pretrained=True)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [26]:
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.efficientnet_b0(pretrained=True)

        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [31]:
#declaring the efficentNetModel b3 structure 
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetModel, self).__init__()
        self.model = models.efficientnet_b3(pretrained=True)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [12]:
# Initialize models, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

#loading the models to the device and declaring them
vit_model = ViTModel(num_classes).to(device)
efficientnet_model = EfficientNetModel(num_classes).to(device)
resnet_model = ResNet18Model(num_classes).to(device)

#loss function and optimizers for each model
criterion = nn.CrossEntropyLoss()
vit_optimizer = optim.Adam(vit_model.parameters(), lr=learning_rate)
efficientnet_optimizer = optim.Adam(efficientnet_model.parameters(), lr=learning_rate)
resnet_optimizer = optim.Adam(resnet_model.parameters(), lr=learning_rate)


cuda


c:\Users\yusle\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\yusle\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Users\yusle\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_W

In [ ]:
def train_model(model, optimizer, train_loader, val_loader, num_epochs, model_name):
    maxAccuracy=80
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Training {model_name} Epoch {epoch+1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader.dataset)
        accuracy = correct / total * 100

        print(f"{model_name} Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {accuracy:.2f}%")
        #saving best model 
        if accuracy > maxAccuracy: 
            torch.save(efficientnet_model.state_dict(), 'efficientnet_model.pth')
            maxAccuracy=accuracy
            print(f"Saved model with best accuracy of {maxAccuracy:.2f}%")
        


In [ ]:
def train_model(model, optimizer, train_loader, val_loader, num_epochs, model_name, num_classes):
    max_mean_accuracy = 79.0

    for epoch in range(num_epochs):

        # ------------------ TRAIN ------------------
        model.train()
        train_loss = 0.0

        for inputs, labels in tqdm(train_loader, desc=f"Training {model_name} Epoch {epoch+1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)

        # ------------------ VALIDATION ------------------
        model.eval()
        val_loss = 0.0

        correct = 0
        total = 0

        # Per-class tracking
        class_correct = torch.zeros(num_classes)
        class_total = torch.zeros(num_classes)

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                # Per-class stats
                for i in range(len(labels)):
                    label = labels[i]
                    class_total[label] += 1
                    if predicted[i] == label:
                        class_correct[label] += 1

        val_loss /= len(val_loader.dataset)
        overall_accuracy = 100 * correct / total

        # Compute per-class accuracy
        class_accuracies = 100 * class_correct / class_total.clamp(min=1)

        # Mean class accuracy (balanced accuracy)
        mean_class_accuracy = class_accuracies.mean().item()

        # ------------------ PRINT METRICS ------------------
        print(f"\n{model_name} Epoch {epoch+1}/{num_epochs}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Overall Accuracy: {overall_accuracy:.2f}%")
        print(f"Mean Class Accuracy: {mean_class_accuracy:.2f}%")

        for i in range(num_classes):
            print(f"Class {i} Accuracy: {class_accuracies[i]:.2f}%")

        # ------------------ SAVE BEST MODEL ------------------
        if mean_class_accuracy > max_mean_accuracy:
            torch.save(model.state_dict(), f"{model_name}_best_whale_{mean_class_accuracy:.2f}%.pth")
            max_mean_accuracy = mean_class_accuracy
            print(f"Saved best model with Mean Class Accuracy: {max_mean_accuracy:.2f}%\n")
        # if class_accuracies[0] > 60 and class_accuracies[1] > 60 and class_accuracies[2] > 60 and class_accuracies[3] > 60:
        #     torch.save(model.state_dict(), f"{model_name}_best_whale_cm{mean_class_accuracy:.2f}%_c1{class_accuracies[0]:.2f}%_c2{class_accuracies[1]:.2f}%_c3{class_accuracies[2]:.2f}%_c4{class_accuracies[3]:.2f}%.pth")
        #     max_mean_accuracy = mean_class_accuracy
        #     print(f"Saved best model with Mean Class Accuracy: {max_mean_accuracy:.2f}%\n")
        


In [51]:

# Train Vision Transformer
train_model(vit_model, vit_optimizer, train_loader, val_loader, num_epochs, "ViT")

Training ViT Epoch 1/1: 100%|██████████| 957/957 [49:41<00:00,  3.12s/it]


ViT Epoch 1/1, Train Loss: 1.0102, Val Loss: 0.9945, Val Accuracy: 57.27%


In [14]:
#train resnet50
train_model(resnet_model, resnet_optimizer, train_loader, val_loader, num_epochs, "ResNet18", num_classes)

Training ResNet18 Epoch 1/10: 100%|██████████| 601/601 [00:30<00:00, 19.79it/s]



ResNet18 Epoch 1/10
Train Loss: 0.1422
Val Loss: 0.0700
Overall Accuracy: 98.08%
Mean Class Accuracy: 98.31%
Class 0 Accuracy: 96.09%
Class 1 Accuracy: 97.60%
Class 2 Accuracy: 99.53%
Class 3 Accuracy: 100.00%
Saved best model with Mean Class Accuracy: 98.31%

Saved best model with Mean Class Accuracy: 98.31%



Training ResNet18 Epoch 2/10: 100%|██████████| 601/601 [00:30<00:00, 19.64it/s]



ResNet18 Epoch 2/10
Train Loss: 0.0478
Val Loss: 0.0389
Overall Accuracy: 98.57%
Mean Class Accuracy: 98.28%
Class 0 Accuracy: 96.65%
Class 1 Accuracy: 100.00%
Class 2 Accuracy: 96.59%
Class 3 Accuracy: 99.90%
Saved best model with Mean Class Accuracy: 98.28%



Training ResNet18 Epoch 3/10: 100%|██████████| 601/601 [00:30<00:00, 19.90it/s]


...


ResNet18 Epoch 8/10
Train Loss: 0.0270
Val Loss: 0.0194
Overall Accuracy: 99.05%
Mean Class Accuracy: 99.08%
Class 0 Accuracy: 96.46%
Class 1 Accuracy: 100.00%
Class 2 Accuracy: 99.84%
Class 3 Accuracy: 100.00%
Saved best model with Mean Class Accuracy: 99.08%

Saved best model with Mean Class Accuracy: 99.08%



Training ResNet18 Epoch 9/10: 100%|██████████| 601/601 [00:30<00:00, 20.03it/s]



ResNet18 Epoch 9/10
Train Loss: 0.0406
Val Loss: 0.0205
Overall Accuracy: 99.05%
Mean Class Accuracy: 99.09%
Class 0 Accuracy: 96.37%
Class 1 Accuracy: 100.00%
Class 2 Accuracy: 100.00%
Class 3 Accuracy: 100.00%
Saved best model with Mean Class Accuracy: 99.09%

Saved best model with Mean Class Accuracy: 99.09%



Training ResNet18 Epoch 10/10: 100%|██████████| 601/601 [00:30<00:00, 19.99it/s]



ResNet18 Epoch 10/10
Train Loss: 0.0239
Val Loss: 0.0181
Overall Accuracy: 99.22%
Mean Class Accuracy: 99.16%
Class 0 Accuracy: 97.58%
Class 1 Accuracy: 100.00%
Class 2 Accuracy: 99.07%
Class 3 Accuracy: 100.00%
Saved best model with Mean Class Accuracy: 99.16%

Saved best model with Mean Class Accuracy: 99.16%



In [1]:
# Train EfficientNet
train_model(efficientnet_model, efficientnet_optimizer, train_loader, val_loader, num_epochs, "EfficientNet", num_classes)


NameError: name 'train_model' is not defined

In [ ]:
# SAVE (after training)
torch.save(efficientnet_model.state_dict(), f'efficientnet_best.pth')



In [ ]:
# Initialize models, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

vit_model = ViTModel(num_classes).to(device)
MyModel = EfficientNetModel(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
vit_optimizer = optim.Adam(vit_model.parameters(), lr=learning_rate)
efficientnet_optimizer = optim.Adam(efficientnet_model.parameters(), lr=learning_rate)

In [50]:
# LOAD (in your Flask app)
model = MyModel()
model.load_state_dict(torch.load('model.pth', map_location='cpu'))
model.eval()

NameError: name 'MyModel' is not defined

In [ ]:
# Example single datapoint
sample = torch.randn(40)

# Add batch dimension (VERY IMPORTANT)
sample = sample.unsqueeze(0)

with torch.no_grad():
    output = model(sample)

predicted_class = torch.argmax(output, dim=1)
print("Prediction:", predicted_class.item())


fix code from here down 

In [7]:
from torchvision import datasets
from torch.utils.data import DataLoader

val_dataset = datasets.ImageFolder(
    root=r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\SplitClasses\Val",
    transform=transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

class_names = val_dataset.classes
num_classes = len(class_names)


NameError: name 'transform' is not defined

In [8]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score


In [9]:
def evaluate_model(model, dataloader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Accuracy
    accuracy = accuracy_score(all_labels, all_preds)

    # Recall (macro)
    recall = recall_score(all_labels, all_preds, average="macro")

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Sensitivity & Specificity
    sensitivity = []
    specificity = []

    for i in range(len(cm)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        sens = TP / (TP + FN) if (TP + FN) > 0 else 0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0

        sensitivity.append(sens)
        specificity.append(spec)

    results = {
        "accuracy": accuracy,
        "recall_macro": recall,
        "sensitivity_per_class": sensitivity,
        "specificity_per_class": specificity,
        "sensitivity_macro": np.mean(sensitivity),
        "specificity_macro": np.mean(specificity),
        "confusion_matrix": cm
    }

    return results


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

metrics = evaluate_model(model, val_loader, device)


NameError: name 'model' is not defined

In [ ]:
print(f"Accuracy:    {metrics['accuracy']:.4f}")
print(f"Recall:      {metrics['recall_macro']:.4f}")
print(f"Sensitivity: {metrics['sensitivity_macro']:.4f}")
print(f"Specificity: {metrics['specificity_macro']:.4f}")

print("\nPer-class metrics:")
for i, class_name in enumerate(class_names):
    print(
        f"{class_name:10s} | "
        f"Sensitivity: {metrics['sensitivity_per_class'][i]:.4f} | "
        f"Specificity: {metrics['specificity_per_class'][i]:.4f}"
    )
